## 2026 EY AI & Data Challenge - Landsat Data Extraction Notebook

This notebook demonstrates Landsat data extraction and the creation of an output file to be used by the benchmark notebook. The baseline data is [Landsat Collection 2 Level 2](https://planetarycomputer.microsoft.com/dataset/landsat-c2-l2) data from the MS Planetary Computer catalog. 

<b>Caution</b> ... This notebook requires significant execution time as there are 9319 data points (unique locations and times) used for data extraction from the Landsat archive. The code takes about 7 hours to run to completion on a typical laptop computer and typical internet connection. Lower execution times are likely possible with optimization of the data extraction process and use of cloud computing services. 

### Load Python Dependencies

In [9]:
import warnings
warnings.filterwarnings("ignore")

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load
from pystac.extensions.eo import EOExtension as eo

from datetime import date
from tqdm import tqdm
import os
import hashlib

<h3>Extracting Landsat Data Using API Calls</h3> <p align="justify"> The API-based method allows us to efficiently access <b>Landsat</b> data for specific coordinates and time periods, ensuring scalability and reproducibility of the process. </p> <p align="justify"> Through the API, we can query individual bands or compute indices like <b>NDMI</b> on-the-fly. This approach reduces storage requirements and simplifies data preprocessing, making it ideal for large-scale environmental and water quality analysis. </p>

<p>The <b>compute_Landsat_values</b> function extracts Landsat surface reflectance values for specific sampling locations using a 100 m focal buffer around each point. For each location:</p>

<ul>
  <li>A bounding box (bbox) is created around the latitude and longitude coordinates.</li>
  <li>The Microsoft Planetary Computer API is queried for Landsat-8 Level-2 surface reflectance imagery within the date range.</li>
  <li>The nearest low-cloud (<10% cloud cover) scene is selected, and the specified bands (<b>red</b>, <b>green</b>, <b>nir08</b>, <b>swir16</b>, <b>swir22</b>) are loaded.</li>
  <li>Median values of the pixels within the bounding box are computed to reduce the effect of noise or outliers.</li>
</ul>

<p><b>Why the buffer value is 0.00089831:</b></p>

<p>We want a ~100 m buffer around each point. At the equator, 1 degree ≈ 110 km. Therefore, the degree equivalent of 100 m is:</p>

<p style="text-align:center;">
  <em>buffer_deg = 100 m / 110,000 m/deg ≈ 0.00089831</em>
</p>

<p>This slightly adjusted value ensures that the buffer approximately matches the pixel resolution of Landsat imagery, capturing a ~100 m area around each sampling location.</p>


In [10]:
# Setup
tqdm.pandas()

# Create directory for saving image patches
IMAGE_DIR = "../data/processed/landsat_images"
os.makedirs(IMAGE_DIR, exist_ok=True)

def compute_Landsat_values(row, save_images=True):
    """
    Extract Landsat values and optionally save image patches for CNN training.
    
    Parameters:
    -----------
    row : pandas.Series
        Row containing Latitude, Longitude, and Sample Date
    save_images : bool
        If True, save multi-channel image patches as .npy files
        
    Returns:
    --------
    pandas.Series with extracted features and image path
    """
    lat = row['Latitude']
    lon = row['Longitude']
    date = pd.to_datetime(row['Sample Date'], dayfirst=True, errors='coerce')

    # Buffer size for ~100m 
    bbox_size = 0.00089831  
    bbox = [
        lon - bbox_size / 2,
        lat - bbox_size / 2,
        lon + bbox_size / 2,
        lat + bbox_size / 2
    ]

    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )

    # Wider search range, we'll filter to nearest date later
    search = catalog.search(
        collections=["landsat-c2-l2"],
        bbox=bbox,
        datetime="2011-01-01/2015-12-31",
        query={"eo:cloud_cover": {"lt": 10}},
    )
    
    items = search.item_collection()

    if not items:
        return pd.Series({
            "Latitude": lat, "Longitude": lon, "Sample Date": date, 
            "red": np.nan
        })

    try:
        # Convert sample date to UTC
        sample_date_utc = date.tz_localize("UTC") if date.tzinfo is None else date.tz_convert("UTC")

        # Pick the item closest to the sample date
        items = sorted(
            items,
            key=lambda x: abs(pd.to_datetime(x.properties["datetime"]).tz_convert("UTC") - sample_date_utc)
        )
        selected_item = pc.sign(items[0])

        # Load required bands (including red for NDCI and NDTI)
        bands_of_interest = ["red"]
        data = stac_load([selected_item], bands=bands_of_interest, bbox=bbox).isel(time=0)

        red = data["red"].astype("float")
        
        # Compute medians
        median_red = float(red.median(skipna=True).values)

        # Replace 0 with NaN
        median_red = median_red if median_red != 0 else np.nan
        
        return pd.Series({
            "Latitude": lat, 
            "Longitude": lon, 
            "Sample Date": date, 
            "red": median_red,
        })
    
    except Exception as e:
        return pd.Series({
            "Latitude": lat, "Longitude": lon, "Sample Date": date, 
            "red": np.nan
        })


In [11]:
# Setup
def load_row(df_source):
    tqdm.pandas()

    row = df_source.loc[0]
    lat = row['Latitude']
    lon = row['Longitude']
    date = pd.to_datetime(row['Sample Date'], dayfirst=True, errors='coerce')

    # Buffer size for ~100m 
    bbox_size = 0.00089831  
    bbox = [
        lon - bbox_size / 2,
        lat - bbox_size / 2,
        lon + bbox_size / 2,
        lat + bbox_size / 2
    ]

    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )

    # Wider search range, we'll filter to nearest date later
    search = catalog.search(
        collections=["landsat-c2-l2"],
        bbox=bbox,
        datetime="2011-01-01/2015-12-31",
        query={"eo:cloud_cover": {"lt": 10}},
    )

    items = search.item_collection()

    try:
        # Convert sample date to UTC
        sample_date_utc = date.tz_localize("UTC") if date.tzinfo is None else date.tz_convert("UTC")

        # Pick the item closest to the sample date
        items = sorted(
            items,
            key=lambda x: abs(pd.to_datetime(x.properties["datetime"]).tz_convert("UTC") - sample_date_utc)
        )
        selected_item = pc.sign(items[0])
            
        bands_of_interest = ["red"]
        data = stac_load([selected_item], bands=bands_of_interest, bbox=bbox).isel(time=0)

        red = data["red"].astype("float")
        # Compute medians
        median_red = float(red.median(skipna=True).values)
        return median_red
            

    except Exception as e:
        pass


In [12]:

import time
for i in range(3):
    start = time.perf_counter()
    load_row(landsat_df)
    end = time.perf_counter()
    elapsed_time = end - start
    print(f"Execution time: {elapsed_time:.4f} seconds")

    

Execution time: 1.7819 seconds
Execution time: 2.0870 seconds
Execution time: 2.0928 seconds


### Extracting features for the training dataset

In [13]:
Water_Quality_df=pd.read_csv('../data/original/landsat_features_training.csv')
Water_Quality_df.head()

,Latitude,Longitude,Sample Date,nir,green,swir16,swir22,NDMI,MNDWI
0,-28.760833,17.730278,02-01-2011,11190.0,11426.0,7687.5,7645.0,0.185538,0.195595
1,-26.861111,28.884722,03-01-2011,17658.5,9550.0,13746.5,10574.0,0.124566,-0.180134
2,-26.450000,28.085833,03-01-2011,15210.0,10720.0,17974.0,14201.0,-0.083293,-0.252805
3,-27.671111,27.236944,03-01-2011,14887.0,10943.0,13522.0,11403.0,0.048048,-0.105416
4,-27.356667,27.286389,03-01-2011,16828.5,9502.5,12665.5,9643.0,0.141147,-0.142683


In [14]:
Water_Quality_df.shape

(9319, 9)

In [18]:
Water_Quality_df_200_999 = Water_Quality_df.loc[200:999]
Water_Quality_df_200_999.shape

(800, 9)

<h3>Note:</h3>
<p>The Landsat data extraction process for all 9,319 locations typically requires 7+ hours when executed in a single run. During long executions, you may occasionally encounter API limits, timeout errors, or request failures. To avoid these interruptions, we recommend running the extraction in smaller batches.</p>

<p>In this notebook, we provide a sample code snippet demonstrating how to extract data for the first 200 locations. Participants are encouraged to follow the same batching approach to extract data for all 9,319 locations safely and efficiently.</p>

<p>We have already executed the full extraction for all 9,319 locations and saved the output to <b>landsat_features_training.csv</b>, which will be used in the benchmark notebook.
Similarly, participants can extract Landsat features in batches, combine the batch outputs, and save the final merged dataset as <b>landsat_features_training.csv</b> to ensure the benchmark notebook runs smoothly.</p>

In [19]:
# Extract band values from Landsat for training dataset
train_features_path = "../data/processed/landsat_features_training_200_red.csv"

print("🚀 Running Landsat feature extraction for training data...")
landsat_train_features = Water_Quality_df_200_999.progress_apply(compute_Landsat_values, axis=1)
landsat_train_features.to_csv(train_features_path, index=False)

🚀 Running Landsat feature extraction for training data...


100%|██████████| 800/800 [25:41<00:00,  1.93s/it]


<p><b>Water Quality Indices:</b></p>
<p>In this notebook, we compute several commonly used water-related indices from the extracted Landsat bands:</p>
<ul>
  <li><b>NDMI (Normalized Difference Moisture Index):</b> Measures vegetation water content and surface moisture. Computed as <em>(NIR - SWIR16) / (NIR + SWIR16)</em>.</li>
  <li><b>MNDWI (Modified Normalized Difference Water Index):</b> Highlights open water features by enhancing water reflectance and suppressing built-up areas. Computed as <em>(Green - SWIR16) / (Green + SWIR16)</em>.</li>
  <li><b>NDCI (Normalized Difference Chlorophyll Index):</b> Used to detect chlorophyll-a concentration in water bodies, which is an indicator of algal blooms and water quality. Computed as <em>(NIR - Red) / (NIR + Red)</em>.</li>
  <li><b>NDTI (Normalized Difference Turbidity Index):</b> Measures water turbidity, which indicates the presence of suspended particles in water. Computed as <em>(Red - Green) / (Red + Green)</em>.</li>
  <li><b>NDSI (Normalized Difference Snow Index):</b> Originally designed for snow detection, but also useful for water quality analysis. Computed as <em>(Green - SWIR16) / (Green + SWIR16)</em>.</li>
</ul>

<p>An <b>epsilon value</b> (<em>eps = 1e-10</em>) is added in the denominators to avoid division by zero. These indices are widely used in hydrological and water quality analyses for detecting water presence, vegetation moisture levels, chlorophyll content, turbidity, and other water quality parameters.</p>


In [ ]:
# Create indices: NDMI, MNDWI, NDCI, NDTI, and NDSI
eps = 1e-10
landsat_train_features['NDMI'] = (landsat_train_features['nir'] - landsat_train_features['swir16']) / (landsat_train_features['nir'] + landsat_train_features['swir16'] + eps)
landsat_train_features['MNDWI'] = (landsat_train_features['green'] - landsat_train_features['swir16']) / (landsat_train_features['green'] + landsat_train_features['swir16'] + eps)
landsat_train_features['NDCI'] = (landsat_train_features['nir'] - landsat_train_features['red']) / (landsat_train_features['nir'] + landsat_train_features['red'] + eps)
landsat_train_features['NDTI'] = (landsat_train_features['red'] - landsat_train_features['green']) / (landsat_train_features['red'] + landsat_train_features['green'] + eps)
landsat_train_features['NDSI'] = (landsat_train_features['green'] - landsat_train_features['swir16']) / (landsat_train_features['green'] + landsat_train_features['swir16'] + eps)

### Automated batched extraction (optional)

Run extraction in configurable batches. Each batch is saved to a separate CSV (e.g. `landsat_features_training_0_199_red.csv`) so you can stop and resume. Set `resume=True` to skip batches that already have output files. When all batches are done, run the merge cell below to combine them into one file.

**If you already ran a chunk** (e.g. rows 200–999): save or rename that CSV as `landsat_features_training_200_999_red.csv` so the merge step will include it. To only run the remaining rows, set `START_ROW = 1000` and run the batch cell, then merge.

In [20]:
# --- Batched extraction config (run after Water_Quality_df is loaded) ---
BATCH_SIZE = 200                    # rows per batch (e.g. 200 or 800)
START_ROW = 1000                       # first row index (inclusive)
END_ROW = len(Water_Quality_df) - 1  # last row index (inclusive); use e.g. 999 to only run up to row 999
OUTPUT_DIR = "../data/processed"
BATCH_PREFIX = "landsat_features_training"  # batch files: {prefix}_{start}_{end}_red.csv
resume = True                       # if True, skip batches that already have a saved CSV

os.makedirs(OUTPUT_DIR, exist_ok=True)
n_total = END_ROW - START_ROW + 1
n_batches = (n_total + BATCH_SIZE - 1) // BATCH_SIZE
print(f"Total rows: {n_total} (rows {START_ROW}–{END_ROW}) in {n_batches} batch(es) of up to {BATCH_SIZE}")

for b in range(n_batches):
    start = START_ROW + b * BATCH_SIZE
    end = min(START_ROW + (b + 1) * BATCH_SIZE - 1, END_ROW)
    batch_path = os.path.join(OUTPUT_DIR, f"{BATCH_PREFIX}_{start}_{end}_red.csv")
    if resume and os.path.exists(batch_path):
        print(f"⏭ Skipping batch {b+1}/{n_batches} (rows {start}–{end}): already exists")
        continue
    batch_df = Water_Quality_df.loc[start:end]
    print(f"🚀 Batch {b+1}/{n_batches}: rows {start}–{end} ({len(batch_df)} rows)...")
    features = batch_df.progress_apply(compute_Landsat_values, axis=1)
    features.to_csv(batch_path, index=False)
    print(f"   ✅ Saved to {batch_path}")

Total rows: 8319 (rows 1000–9318) in 42 batch(es) of up to 200
🚀 Batch 1/42: rows 1000–1199 (200 rows)...


  3%|▎         | 6/200 [00:44<23:53,  7.39s/it]


APIError: <!DOCTYPE html PUBLIC '-//W3C//DTD XHTML 1.0 Transitional//EN' 'http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd'>
<html xmlns='http://www.w3.org/1999/xhtml'>

<head>
    <meta content='text/html; charset=utf-8' http-equiv='content-type' />
    <style type='text/css'>
        body {
            font-family: Arial;
            margin-left: 40px;
        }

        img {
            border: 0 none;
        }

        #content {
            margin-left: auto;
            margin-right: auto
        }

        #message h1 {
            font-size: 24px;
            font-weight: normal;
            color: #000000;
            margin: 34px 0px 0px 0px
        }

        #message h2 {
            font-size: 20px;
            font-weight: normal;
            color: #000000;
            margin: 34px 0px 0px 0px
        }

        #message p {
            font-size: 16px;
            color: #000000;
            margin: 8px 0px 0px 0px
        }

        #message hr {
            m

### Merge batch outputs into one file

After all batches have been run, run the cell below to concatenate batch CSVs in row order and save a single `landsat_features_training_red.csv`. Uses the same `START_ROW`, `END_ROW`, `BATCH_SIZE`, `OUTPUT_DIR`, and `BATCH_PREFIX` as above.

In [ ]:
# Merge all batch CSVs into one file (discovers *_start_end_red.csv and sorts by start)
import glob
import re
merged_path = os.path.join(OUTPUT_DIR, "landsat_features_training_red.csv")
pattern = os.path.join(OUTPUT_DIR, f"{BATCH_PREFIX}_*_*_red.csv")
batch_files = glob.glob(pattern)
# Parse start/end from filename (e.g. ..._200_999_red.csv) and sort by start
def start_end(path):
    name = os.path.basename(path)
    m = re.match(r".*_(\d+)_(\d+)_red\.csv", name)
    return (int(m.group(1)), int(m.group(2))) if m else (0, 0)
batch_files.sort(key=start_end)
parts = [pd.read_csv(p) for p in batch_files]
if parts:
    merged = pd.concat(parts, ignore_index=True)
    merged.to_csv(merged_path, index=False)
    print(f"✅ Merged {len(parts)} batch(es), {len(merged)} rows → {merged_path}")
else:
    print("No batch files found to merge.")

In [ ]:
landsat_train_features['Latitude'] = Water_Quality_df_200['Latitude']
landsat_train_features['Longitude'] = Water_Quality_df_200['Longitude']
landsat_train_features['Sample Date'] = Water_Quality_df_200['Sample Date']
landsat_train_features = landsat_train_features[['Latitude', 'Longitude', 'Sample Date', 'red', 'nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI', 'NDCI', 'NDTI', 'NDSI', 'image_path']]

In [ ]:
landsat_train_features.to_csv(train_features_path, index=False)

In [ ]:
# Preview File
landsat_train_features.head()

,Latitude,Longitude,Sample Date,nir,green,swir16,swir22,NDMI,MNDWI
0,-28.760833,17.730278,02-01-2011,11190.0,11426.0,7687.5,7645.0,0.185538,0.195595
1,-26.861111,28.884722,03-01-2011,17658.5,9550.0,13746.5,10574.0,0.124566,-0.180134
2,-26.450000,28.085833,03-01-2011,15210.0,10720.0,17974.0,14201.0,-0.083293,-0.252805
3,-27.671111,27.236944,03-01-2011,14887.0,10943.0,13522.0,11403.0,0.048048,-0.105416
4,-27.356667,27.286389,03-01-2011,16828.5,9502.5,12665.5,9643.0,0.141147,-0.142683


### Extracting features for the validation dataset

In [ ]:
Validation_df=pd.read_csv('submission_template.csv')
Validation_df.head()

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-32.043333,27.822778,01-09-2014,NaN,NaN,NaN
1,-33.329167,26.077500,16-09-2015,NaN,NaN,NaN
2,-32.991639,27.640028,07-05-2015,NaN,NaN,NaN
3,-34.096389,24.439167,07-02-2012,NaN,NaN,NaN
4,-32.000556,28.581667,01-10-2014,NaN,NaN,NaN


In [ ]:
Validation_df.shape

(200, 6)

In [ ]:
# Extract band values from Landsat for submission dataset
val_features_path = "landsat_features_validation.csv"

print("🚀 Running Landsat feature extraction for validation data...")
landsat_val_features = Validation_df.progress_apply(compute_Landsat_values, axis=1)
landsat_val_features.to_csv(val_features_path, index=False)

🚀 Running Landsat feature extraction for validation data...


100%|██████████| 200/200 [10:38<00:00,  3.19s/it]


In [ ]:
# Create indices: NDMI, MNDWI, NDCI, NDTI, and NDSI
eps = 1e-10
landsat_val_features['NDMI'] = (landsat_val_features['nir'] - landsat_val_features['swir16']) / (landsat_val_features['nir'] + landsat_val_features['swir16'] + eps)
landsat_val_features['MNDWI'] = (landsat_val_features['green'] - landsat_val_features['swir16']) / (landsat_val_features['green'] + landsat_val_features['swir16'] + eps)
landsat_val_features['NDCI'] = (landsat_val_features['nir'] - landsat_val_features['red']) / (landsat_val_features['nir'] + landsat_val_features['red'] + eps)
landsat_val_features['NDTI'] = (landsat_val_features['red'] - landsat_val_features['green']) / (landsat_val_features['red'] + landsat_val_features['green'] + eps)
landsat_val_features['NDSI'] = (landsat_val_features['red'] - landsat_val_features['nir']) / (landsat_val_features['red'] + landsat_val_features['nir'] + eps)

In [ ]:
landsat_val_features['Latitude'] = Validation_df['Latitude']
landsat_val_features['Longitude'] = Validation_df['Longitude']
landsat_val_features['Sample Date'] = Validation_df['Sample Date']
landsat_val_features = landsat_val_features[['Latitude', 'Longitude', 'Sample Date', 'red', 'nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI', 'NDCI', 'NDTI', 'NDSI', 'image_path']]

In [ ]:
landsat_val_features.to_csv(val_features_path, index=False)

In [ ]:
# Preview File
landsat_val_features.head()

,Latitude,Longitude,Sample Date,nir,green,swir16,swir22,NDMI,MNDWI
0,-32.043333,27.822778,01-09-2014,15229.0,12868.0,14797.0,12421.0,0.014388,-0.069727
1,-33.329167,26.077500,16-09-2015,NaN,NaN,NaN,NaN,NaN,NaN
2,-32.991639,27.640028,07-05-2015,16221.0,9304.5,12536.5,9958.0,0.128123,-0.147979
3,-34.096389,24.439167,07-02-2012,NaN,NaN,NaN,NaN,NaN,NaN
4,-32.000556,28.581667,01-10-2014,9125.0,11100.5,9455.0,8711.0,-0.017761,0.080052
